# Part 1 – Machine Learning models for network intrusion detection (CICIDS2017)

Implements the first part of the report's methodology:

* **Dataset** – a combination of CICIDS2017 subsets with **DDoS, PortScan and DoS** attacks alongside normal (BENIGN) traffic.
  The class mix follows the class supports shown in Fig 1 / Fig 2 of the report (BENIGN 114864, DDoS 25604, PortScan 35756, DoS 18054).
* **Preprocessing** – missing / invalid values removed; feature engineering (**packets-per-second**, **forward/backward packet ratio**,
  **window size difference**); **LabelEncoder** for labels; feature selection based on **correlation** and **information gain**;
  **StandardScaler**.
* **Tier 1** – Logistic Regression, Decision Tree, Random Forest, XGBoost.
* **Tier 2** – ensemble of Random Forest and XGBoost with **soft voting**.
* **Evaluation** – accuracy, precision, recall, F1 score, training time, inference time, confusion matrices (Seaborn heatmaps).

Outputs: `results/part1_ml_results.csv`, `figures/part1_*.png`.

In [ ]:
import os, time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

SEED = 42
DATA_DIR = "datasets/cicids2017/MachineLearningCVE"
os.makedirs("results", exist_ok=True); os.makedirs("figures", exist_ok=True)

## 1. Curate the dataset – DDoS, PortScan and DoS subsets with normal traffic

In [ ]:
subsets = ["Wednesday-workingHours", "Friday-WorkingHours-Afternoon-DDos", "Friday-WorkingHours-Afternoon-PortScan"]
df = pd.concat([pd.read_csv(f"{DATA_DIR}/{s}.pcap_ISCX.csv", encoding="latin-1", low_memory=False) for s in subsets],
               ignore_index=True)
df.columns = df.columns.str.strip()
# The four DoS tools of the Wednesday capture are grouped into one 'DoS' class (as in Fig 1); Heartbleed (11 flows) is not part of this study.
df["Label"] = df["Label"].replace({"DoS Hulk": "DoS", "DoS GoldenEye": "DoS", "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS"})
df = df[df["Label"].isin(["BENIGN", "DDoS", "PortScan", "DoS"])]
print(df["Label"].value_counts())

## 2. Preprocessing – remove missing / invalid values

In [ ]:
num = df.drop(columns=["Label"]).apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
valid = num.notna().all(axis=1)
print("rows removed (missing / invalid):", int((~valid).sum()))
df = pd.concat([num[valid], df.loc[valid, "Label"]], axis=1).reset_index(drop=True)
df = df.loc[:, ~df.columns.duplicated()]          # 'Fwd Header Length.1' duplicates 'Fwd Header Length'

In [ ]:
# Class mix of the curated dataset follows the class supports in Fig 1 of the report.
fig1_support = {"BENIGN": 114864, "DDoS": 25604, "PortScan": 35756, "DoS": 18054}
avail = df["Label"].value_counts()
scale = min(avail[c] / n for c, n in fig1_support.items())
parts = [df[df["Label"] == c].sample(n=int(round(n * scale)), random_state=SEED) for c, n in fig1_support.items()]
df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(df["Label"].value_counts(), "\ntotal:", len(df))

## 3. Feature engineering – packets-per-second, forward/backward packet ratio, window size difference

In [ ]:
dur_s = df["Flow Duration"] / 1e6                              # Flow Duration is in microseconds
total_pkts = df["Total Fwd Packets"] + df["Total Backward Packets"]
df["Packets Per Second"] = np.where(dur_s > 0, total_pkts / dur_s.where(dur_s > 0, 1), 0.0)
df["Fwd Bwd Packet Ratio"] = df["Total Fwd Packets"] / (df["Total Backward Packets"] + 1)
df["Window Size Difference"] = df["Init_Win_bytes_forward"] - df["Init_Win_bytes_backward"]
df[["Packets Per Second", "Fwd Bwd Packet Ratio", "Window Size Difference"]].describe().T

## 4. Label encoding

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["Label"])
class_names = list(label_encoder.classes_)
X = df.drop(columns=["Label"])
print(dict(enumerate(class_names)), X.shape)

## 5. Feature selection – correlation and information gain

1. **Correlation** – constant features are dropped, and of every pair of features with |Pearson r| > 0.95 only one is kept.
2. **Information gain** – mutual information between each remaining feature and the label; the features with the highest
   information gain are retained.

In [ ]:
X = X.loc[:, X.nunique() > 1]
corr = X.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
drop_corr = [c for c in upper.columns if (upper[c] > 0.95).any()]
X = X.drop(columns=drop_corr)
print("dropped (correlation):", len(drop_corr), "-> remaining:", X.shape[1])

N_IG = 25
sample = X.sample(n=min(100_000, len(X)), random_state=SEED)
ig = pd.Series(mutual_info_classif(sample, y[sample.index], random_state=SEED), index=X.columns).sort_values(ascending=False)
selected = ig.index[:N_IG].tolist()
print("selected by information gain:", selected)
ig.head(N_IG).plot.barh(figsize=(7, 7)).invert_yaxis(); plt.title("Information gain of selected features"); plt.tight_layout()
plt.savefig("figures/part1_information_gain.png", dpi=120); plt.show()
X = X[selected]

## 6. Train / test split and StandardScaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X.values, y, test_size=0.2, random_state=SEED, stratify=y)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train); X_test = scaler.transform(X_test)
print("train:", X_train.shape, "test:", X_test.shape)

## 7. Tier 1 (classical ML) and Tier 2 (RF + XGBoost soft-voting ensemble)

Training time is the wall-clock time of `fit`; inference time is the average time to classify one flow
(prediction time over the whole test set divided by the number of test flows), in milliseconds.

In [ ]:
models = {
    "M1 (Random Forest)": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    "M2 (Decision Tree)": DecisionTreeClassifier(random_state=SEED),
    "M3 (XGBoost)": XGBClassifier(n_estimators=200, n_jobs=-1, random_state=SEED, eval_metric="mlogloss"),
    "M4 (Logistic Regression)": LogisticRegression(max_iter=1000),
    "H1 (RF + XGBoost)": VotingClassifier(
        estimators=[("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)),
                    ("xgb", XGBClassifier(n_estimators=200, n_jobs=-1, random_state=SEED, eval_metric="mlogloss"))],
        voting="soft"),
}
rows, preds = [], {}
for name, model in models.items():
    t0 = time.perf_counter(); model.fit(X_train, y_train); train_s = time.perf_counter() - t0
    t0 = time.perf_counter(); p = model.predict(X_test); infer_ms = (time.perf_counter() - t0) / len(X_test) * 1000
    preds[name] = p
    rows.append({"Model": name, "Accuracy": accuracy_score(y_test, p),
                 "Precision": precision_score(y_test, p, average="macro"), "Recall": recall_score(y_test, p, average="macro"),
                 "F1 Score": f1_score(y_test, p, average="macro"),
                 "Training Time (s)": round(train_s, 2), "Inference Time (ms)": round(infer_ms, 4)})
    print(f"{name}: accuracy={rows[-1]['Accuracy']:.4f}  train={train_s:.1f}s")

## 8. Results – Table 1 (Tier 1) and Table 2 (Tier 2 ensemble)

In [ ]:
results = pd.DataFrame(rows)
pct = results.copy()
for c in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    pct[c] = (pct[c] * 100).map("{:.2f}%".format)
results.to_csv("results/part1_ml_results.csv", index=False)
print("Table 1 - Performance metrics of Machine learning models")
display(pct[pct["Model"].str.startswith("M")].reset_index(drop=True))
print("Table 2 - Performance metrics of Ensemble model")
display(pct[pct["Model"].str.startswith("H")].reset_index(drop=True))

## 9. Model evaluation – classification reports (Fig 1 Random Forest, Fig 2 XGBoost)

In [ ]:
for name, fig in [("M1 (Random Forest)", "Fig 1 - Random Forest"), ("M3 (XGBoost)", "Fig 2 - XGBoost")]:
    print(f"{fig} Classifier Model Evaluation:")
    print(f"Accuracy: {accuracy_score(y_test, preds[name]):.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, preds[name], target_names=class_names, digits=4))

## 10. Confusion matrices (Seaborn heatmaps)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (name, p) in zip(axes.ravel(), preds.items()):
    sns.heatmap(confusion_matrix(y_test, p), annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.savefig("figures/part1_confusion_matrices.png", dpi=120); plt.show()